In [11]:
import pandas as pd
import pandas as pd
import jax
import jax.numpy as jnp
from jax import jit, grad, vmap, pmap
import math
import orbax.checkpoint as ocp
from functools import partial
import os
import gc

jax.config.update("jax_platform_name", "cpu")

os.environ["JAX_LOG_LEVEL"] = "0" 

jax.config.update("jax_default_matmul_precision", "default") 


gc.collect()

print("aye 1")

gold_second_futures = pd.read_csv('GC2026_daily.csv')

gold_second_futures['time'] = pd.to_datetime(
    gold_second_futures['time']
)

gold_second_futures['timestamp'] = (
    gold_second_futures['time'].astype('int64') // 10**9
)


gold_second_futures['timestamp'] = (
    (gold_second_futures['timestamp'] - gold_second_futures['timestamp'].min())/ (gold_second_futures['timestamp'].max() - gold_second_futures['timestamp'].min())
)


X = gold_second_futures[
    ["timestamp", "volume"]
].to_numpy()

y = gold_second_futures[
    ["close"]
].to_numpy()   # (N, 1)

train_data = int(len(X) * 0.8)

X_train = X[:train_data]
y_train = y[:train_data]

X_test = X[train_data:]
y_test = y[train_data:]


sequence_length = 10
batch_size = 32

print("aye 2")

def create_sequences(X, y, sequence_length):
    X_sequences = []
    y_sequences = []

    for i in range(len(X) - sequence_length + 1):
        X_sequences.append(
            X[i:i + sequence_length]
        )

        y_sequences.append(
            y[i:i + sequence_length]
        )

    return (
        jnp.asarray(X_sequences, dtype=jnp.bfloat16),
        jnp.asarray(y_sequences, dtype=jnp.bfloat16)
    )


X_train_seq, y_train_seq = create_sequences(
    X_train,
    y_train,
    sequence_length
)

X_test_seq, y_test_seq = create_sequences(
    X_test,
    y_test,
    sequence_length
)

print("aye 3")




aye 1
aye 2
aye 3


In [12]:
print("X_train_seq",X_train_seq[0].dtype)
print("y_train_seq",y_train_seq[0].dtype)
print("X_test_seq",X_test_seq[0].dtype)
print("y_test_seq",y_test_seq[0].dtype)


X_train_seq bfloat16
y_train_seq bfloat16
X_test_seq bfloat16
y_test_seq bfloat16


In [13]:
# def adam_optimizer(curr_weights, curr_gradients, iter, layer, is_bias=False):

#     if is_bias:
#         moment_1_curr = (beta_1*(bias_moment_1_history.get(layer,np.zeros(curr_weights.shape)))) + (1-beta_1) * curr_gradients
#         bias_moment_1_history[layer] = moment_1_curr
#     else:
#         moment_1_curr = (beta_1*(moment_1_history.get(layer,np.zeros(curr_weights.shape)))) + (1-beta_1) * curr_gradients
#         moment_1_history[layer] = moment_1_curr

#     hat_moment_1 = moment_1_curr/(1-beta_1**iter)

#     if is_bias:
#         moment_2_curr = beta_2*(bias_moment_2_history.get(layer,np.zeros(curr_weights.shape))) + (1-beta_2) * curr_gradients**2
#         bias_moment_2_history[layer] = moment_2_curr
#     else:
#         moment_2_curr = beta_2*(moment_2_history.get(layer,np.zeros(curr_weights.shape))) + (1-beta_2) * curr_gradients**2
#         moment_2_history[layer] = moment_2_curr

#     hat_moment_2 = moment_2_curr/(1-beta_2**iter)

#     return ((learning_rate/(np.sqrt(hat_moment_2)+epsilon)) * hat_moment_1)

In [14]:
def single_tanh(act):
  res = (jnp.e**act - jnp.e**-act)/(jnp.e**act + jnp.e**-act)
  return res

single_tanh_jit = jit(single_tanh)


def tanh_activation(pre_activations):
  pre_act_shape = pre_activations.shape
  reshaped_pre_acts = jnp.reshape(pre_activations,(-1,))
  tanh_acts = vmap(single_tanh_jit)(reshaped_pre_acts)
  return jnp.asarray(jnp.reshape(tanh_acts,pre_act_shape),dtype=jnp.bfloat16)


In [15]:
def meanAbsoluteLoss(predictions,true_labels):
  return jnp.mean(jnp.abs(predictions-true_labels))

def meanAbsoluteLossDerivation(predictions,true_labels):
  return jnp.sign(predictions-true_labels)/len(predictions)


In [16]:
@jax.jit
def forward(layer_weights,inputs, biases):
  # layer_idx 0 for rnn layer 1, 1 for rnn layer 2, 2 for dense layer 1 which is also output layer
  sequence_length = inputs.shape[1]
  concatted_inputs = jnp.zeros((sequence_length,inputs.shape[0],66),dtype=jnp.bfloat16)
  layer_0_activations = jnp.zeros((sequence_length,inputs.shape[0],layer_weights[0].shape[1]),dtype=jnp.bfloat16)
  layer_1_activations = jnp.zeros((sequence_length,inputs.shape[0],layer_weights[1].shape[1]),dtype=jnp.bfloat16)
  layer_2_activations = jnp.zeros((sequence_length,inputs.shape[0],layer_weights[2].shape[1]),dtype=jnp.bfloat16)
  l0prev=jnp.zeros((32,64),dtype=jnp.bfloat16)
  l1prev=jnp.zeros((32,32),dtype=jnp.bfloat16)
  for i in range(sequence_length):
    # print("seq_iter", i)
    inputs_for_layer_0 = jnp.concatenate([inputs[:,i,:],l0prev],dtype=jnp.bfloat16,axis=1)
    # print("inputs_for_layer_0", inputs_for_layer_0.shape)
    # print("layer_weights[0]", layer_weights[0].shape)
      
    temp_layer_0_activation = tanh_activation((inputs_for_layer_0@layer_weights[0]) +biases[0])
    # print("temp_layer_0_activation", temp_layer_0_activation.shape)
    inputs_for_layer_1 = jnp.concatenate([temp_layer_0_activation,l1prev],dtype=jnp.bfloat16,axis=1)
    # print("inputs_for_layer_1",inputs_for_layer_1)
    temp_layer_1_activation = tanh_activation((inputs_for_layer_1@layer_weights[1]) +biases[1])
    temp_layer_2_activation = (temp_layer_1_activation@layer_weights[2]) +biases[2]
    # print("temp_layer_2_activation",temp_layer_2_activation)
      
    layer_0_activations.at[i].set(jnp.asarray(temp_layer_0_activation,dtype=jnp.bfloat16))
    layer_1_activations.at[i].set(jnp.asarray(temp_layer_1_activation,dtype=jnp.bfloat16))
    layer_2_activations.at[i].set(jnp.asarray(temp_layer_2_activation,dtype=jnp.bfloat16))
    concatted_inputs.at[i].set(jnp.asarray(inputs_for_layer_0,dtype=jnp.bfloat16))
    # breakpoint()
    l0prev,l1prev=temp_layer_0_activation,temp_layer_1_activation
  return concatted_inputs,layer_0_activations, layer_1_activations, layer_2_activations




In [17]:
@partial(jax.jit, static_argnames=['sequence_length'])
def backward(
    layer_weights,
    sequence_length,
    concatted_inputs,
    true_labels,
    layer_0_activations,
    layer_1_activations,
    layer_2_activations
    ):

  accumulated_layer_0_weights_changes = None
  accumulated_layer_1_weights_changes = None
  accumulated_layer_2_weights_changes = None

  accumulated_layer_0_bias_changes = None
  accumulated_layer_1_bias_changes = None
  accumulated_layer_2_bias_changes = None

  layer_0_next_timestep_delta = None
  layer_1_next_timestep_delta = None

  for i in range(sequence_length - 1, -1, -1):
    # print("backprop_iter",i)
    preds = jnp.asarray(layer_2_activations[i],dtype=jnp.bfloat16)
    tru_labels = jnp.asarray(true_labels[:,i],dtype=jnp.bfloat16)
    mae_gradient =  meanAbsoluteLossDerivation(preds, tru_labels)
    layer_2_weights_change = layer_1_activations[i] @ mae_gradient   
    layer_2_bias_change = mae_gradient

    # print("mae_gradient", mae_gradient)
    # print("(1-layer_1_activations[i]**2)", (1-layer_1_activations[i]**2))

    layer_1_activations_gradients = (
                    mae_gradient
                    @
                    layer_weights[2].T
                    
                ).T

    # tiled_layer_1_activations_gradients = jnp.tile(layer_1_activations_gradients, (1, 32))

    layer_1_pre_activations_gradients = (
            layer_1_activations_gradients
            * (1-layer_1_activations[i]**2)
            
    ) # this is also the gradient for bias

    layer_1_weights_change = (
        layer_1_pre_activations_gradients.T
            @ jnp.concat([layer_0_activations[i], layer_1_activations[i-1]], axis=1)
          )

    # print("layer_1_activations_gradients ", layer_1_activations_gradients)
    # print("layer_weights[1] ", layer_weights[1])
    # print("(1-layer_0_activations[i]**2) ", (1-layer_0_activations[i]**2))
    # print("layer_0_activations[i]", layer_0_activations[i])


    layer_0_activations_gradients = (
                layer_1_activations_gradients.T
                @
                layer_weights[1].T
            ).T
    sliced_layer_0_activations_gradients = layer_0_activations_gradients[:layer_weights[0].shape[1],:]
    # tiled_layer_0_activations_gradients = jnp.tile(sliced_layer_0_activations_gradients, (1, 32))
  
    layer_0_pre_activations_gradients = (
        sliced_layer_0_activations_gradients.T
        * (1-layer_0_activations[i]**2)
        
    ) 


    # this is also the gradient for bias
    # print("am i here")


    # print("layer_0_activations_gradients ", layer_0_activations_gradients)
    # print("layer_0_activations[i-1] ", layer_0_activations[i-1])

    layer_0_weights_change = (
            layer_0_pre_activations_gradients.T
             @ concatted_inputs[i]
          )
    
    # print("layer_0_weights_change ", layer_0_weights_change)
        
    if i < sequence_length-1:
      # here we have to handle the latent representation branch errors using BPTT
      input_cutoff_for_latent_repr_layer_1 =  layer_0_activations[i-1].shape[0] - 1
      temp_mask_for_rnn_layer_1_input = jnp.zeros((layer_0_activations[i-1].shape[0],layer_weights[1].shape[0]))

      row_indices = jnp.arange(layer_0_activations[i-1].shape[0], dtype=jnp.bfloat16)[:,jnp.newaxis]

      input_mask_for_rnn_layer_1 = jnp.where(row_indices>input_cutoff_for_latent_repr_layer_1,1,temp_mask_for_rnn_layer_1_input)


      masked_input = jnp.concatenate([layer_0_activations[i],layer_1_activations[i-1]],axis=1) * input_mask_for_rnn_layer_1

      # print("masked_input ",masked_input)
      # print("masked_input.T ",masked_input.T)

      weight_rows_indices = jnp.arange(layer_weights[1].shape[0])[:,jnp.newaxis]
      weight_rows_cutoff = layer_weights[0].shape[0] - 1 
      temp_weights_mask_for_layer_1_recurrect_step = jnp.zeros(layer_weights[1].shape)
      # print("temp_weights_mask_for_layer_1_recurrect_step ", temp_weights_mask_for_layer_1_recurrect_step.shape)
      weights_mask_for_layer_1_recurrect_step = jnp.where(weight_rows_indices>weight_rows_cutoff,1,temp_weights_mask_for_layer_1_recurrect_step)
    
      masked_weights = layer_weights[1] * weights_mask_for_layer_1_recurrect_step

      # print("layer_1_next_timestep_delta ",layer_1_next_timestep_delta)
      # print("(1-layer_1_activations[i]**2) ",(1-layer_1_activations[i]**2))
      # print("(masked_weights.T @ layer_1_next_timestep_delta) ",(masked_weights @ layer_1_next_timestep_delta))

      layer_1_recurrent_step_pre_act_gradients = (masked_weights @ layer_1_next_timestep_delta) * (1-layer_1_activations[i]**2)
      layer_1_recurrent_step_weights_change = layer_1_recurrent_step_pre_act_gradients @ masked_input

      layer_1_recurrent_step_bias_change = layer_1_recurrent_step_pre_act_gradients

      # print("layer_1_recurrent_step_bias_change ", layer_1_recurrent_step_bias_change)
        
      input_cutoff_for_latent_repr_layer_0 =  concatted_inputs[i].shape[0] - layer_weights[0].shape[1] - 1
      temp_mask_for_rnn_layer_0 = jnp.zeros(concatted_inputs[i].shape,dtype=jnp.bfloat16)
      # print("temp_mask_for_rnn_layer_0 ", temp_mask_for_rnn_layer_0)
      input_col_indices = jnp.arange(concatted_inputs[i].shape[1])[jnp.newaxis,:]
      # print("input_col_indices", input_col_indices)
      input_mask_for_rnn_layer_0 = jnp.where(input_col_indices>input_cutoff_for_latent_repr_layer_0,1,temp_mask_for_rnn_layer_0)

      # print("input_mask_for_rnn_layer_0", input_mask_for_rnn_layer_0)
     
      masked_input = concatted_inputs[i-1] * input_mask_for_rnn_layer_0

      # print("layer 0 masked_input ", masked_input)

      layer_0_weight_rows_indices = jnp.arange(layer_weights[0].shape[0])[:,jnp.newaxis]
      weight_rows_cutoff = input_cutoff_for_latent_repr_layer_0
      temp_weights_mask_for_layer_0_recurrect_step = jnp.zeros(layer_weights[0].shape,dtype=jnp.bfloat16)
      weights_mask_for_layer_0_recurrect_step = jnp.where(layer_0_weight_rows_indices>weight_rows_cutoff,1,temp_weights_mask_for_layer_0_recurrect_step)

      masked_weights = layer_weights[0] * weights_mask_for_layer_0_recurrect_step

      # print("layer 0 masked_weights ", masked_weights)
        
      # print("layer_0_next_timestep_delta ", layer_0_next_timestep_delta)

      # print("(1-layer_0_activations[i]**2)", (1-layer_0_activations[i]**2))
        
      layer_0_recurrent_step_pre_act_gradients = (layer_0_next_timestep_delta @ masked_weights.T) * (1-layer_0_activations[i]**2)
      
      # print("layer_0_recurrent_step_pre_act_gradients ", layer_0_recurrent_step_pre_act_gradients)
        
      layer_0_recurrent_step_weights_change = layer_0_recurrent_step_pre_act_gradients @ masked_input

      layer_0_recurrent_step_bias_change = layer_0_recurrent_step_pre_act_gradients

      raise Exception("Stop here for debugging")

    layer_0_next_timestep_delta = layer_0_pre_activations_gradients
    layer_1_next_timestep_delta = layer_1_pre_activations_gradients
    # print(f" iter {i} layer_0_next_timestep_delta {layer_0_next_timestep_delta}")
    # print(f" iter {i} layer_1_next_timestep_delta {layer_1_next_timestep_delta}")

  return (
      accumulated_layer_0_weights_changes,
      accumulated_layer_1_weights_changes,
      accumulated_layer_2_weights_changes,
      accumulated_layer_0_bias_changes,
      accumulated_layer_1_bias_changes,
      accumulated_layer_2_bias_changes
  )




In [18]:
def he_initialization(layer_shapes):
    key = jax.random.key(1337)
    key, w_key = jax.random.split(key)
    layer_weights=[]
    layer_biases=[]

    # Define a He/Kaiming normal initializer (excellent for ReLU activations)
    initializer = jax.nn.initializers.he_normal()

    # Initialize the array
    for i, shape in enumerate(layer_shapes):
        layer_weights.append(initializer(w_key, shape, jnp.bfloat16))
        layer_biases.append(jnp.zeros(shape[-1],dtype=jnp.bfloat16))

    return layer_weights,layer_biases

In [19]:
def training_loop():
  layer_weights_shapes = [(66,64),(96,32),(32,1)]
  layer_weights,layer_biases = he_initialization(layer_weights_shapes)

  batch_size = 32
  len_train_samples = len(X_train_seq)
  iters_per_epoch = math.ceil(len_train_samples / batch_size)
  epochs=10

  for epoch in range(epochs):
    for iter in range(iters_per_epoch):
      batch_start = iter * batch_size
      batch_end = (iter + 1) * batch_size
      batch_X = X_train_seq[batch_start:batch_end]
      batch_y = y_train_seq[batch_start:batch_end]
      concatted_inputs,layer_0_activations, layer_1_activations, layer_2_activations = forward(layer_weights,batch_X,layer_biases)

      # Fix: Transpose predictions from (Seq, Batch, 1) to (Batch, Seq, 1) to match batch_y
      preds_transposed = jnp.transpose(layer_2_activations, (1, 0, 2))
      loss = meanAbsoluteLoss(preds_transposed, batch_y)

      (layer_0_weights_change, 
       layer_1_weights_change, 
       layer_2_weights_change, 
       layer_0_bias_changes, 
       layer_1_bias_changes, 
       layer_2_bias_changes ) = backward(
         layer_weights,
         sequence_length,
         concatted_inputs,
         batch_y,
         layer_0_activations, 
         layer_1_activations, 
         layer_2_activations
         )

  # Create a checkpoint manager or direct saver
  ackp = ocp.StandardCheckpointer()
  # Save parameters dictionary/tree to a path
  ackp.save('/content/rnn_weights', args=ocp.args.StandardSave((layer_weights,layer_biases)))

In [21]:
training_loop()

TypeError: mul got incompatible shapes for broadcasting: (96, 32), (32, 32).